# ERA5 Autoencoder
This will demonstrate 

### Import libraries

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import pathlib
import datetime
import functools
import math

In [4]:
import numpy

In [5]:
import xarray

In [6]:
import matplotlib
import cartopy.crs

In [7]:
import site_archive_jasmin

ROOT_DIRECTORIES: {'ERA5lowres': '/gws/ssde/j25b/mohc_shared/dscop/weatherbench/5.625deg/', 'MOGLOBAL': '/gws/ssde/j25b/mohc_shared/dscop/mo_pet_site_archive/mo_global/', 'MOUKV': '/gws/ssde/j25b/mohc_shared/dscop/mo_pet_site_archive/mo_ukv/', 'Himawari': '/gws/ssde/j25b/mohc_shared/mh_pet/rv74_himawari', 'HimawariChannels': '/gws/ssde/j25b/mohc_shared/mh_pet/ra22_himawari', 'Rainfields3': '/gws/ssde/j25b/mohc_shared/mh_pet/rq0Radar', 'ew4_imerg_precip': '/gws/nopw/j04/ew4energy/imerg_2025_summer', 'ew4_merra2_meteo': '/gws/nopw/j04/ew4energy/West_Africa_Merra-2_2015-2025_meteo', 'ew4_merra2_aero': '/gws/nopw/j04/ew4energy/West_Africa_Merra-2_2015-2025_aerosols/3d', 'ew4_mtg_li': '/gws/nopw/j04/ew4energy/MTG_LI/', 'ew4_era5': '/gws/nopw/j04/ew4energy/ERA5/tutorial_202606'}


In [8]:
import pyearthtools

In [9]:
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe

In [10]:
from pyearthtools.data import Petdt, TimeDelta
from pyearthtools.data.exceptions import DataNotFoundError
from pyearthtools.data.indexes import ArchiveIndex, decorators
from pyearthtools.data.transforms import Transform, TransformCollection
from pyearthtools.data.archive import register_archive


In [11]:
from site_archive_jasmin.utilities import (
    cached_exists,
    cached_iterdir,
)  # Could these be moved into a generic module?


In [12]:
import torch

In [13]:
username = os.environ['JUPYTERHB)ER']

NameError: name 'os' is not defined

In [ ]:
mohc_gws_dir = pathlib.Path('/gws/ssde/j25b/mohc_shared/')
ew4_gws_user_dir = ew4_gws_dir / 'user' / 'shaddad'
ew4_user_cache_dir = ew4_gws_user_dir / 'cache'
ew4_imerg_cache_dir = ew4_user_cache_dir / 'imerg'
imerg_norm_dir = ew4_imerg_cache_dir / 'norms'
ew4_imerg_ml_prepped_dir = ew4_imerg_cache_dir / 'ml_prepped'

In [ ]:
era5_accessor = pyearthtools.data.archive.ERA5lowres(["2m_temperature","temperature", "specific_humidity","u", "v"])
era5_accessor

In [ ]:
era5['2015-7-19 00:00']

In [ ]:
era5_prep = petpipe.Pipeline(
    era5_accessor,
    pyearthtools.pipeline.modifications.Cache(
        user_cache_dir,
        pattern_kwargs={'extension': 'nc'},
        cache_validity='override',
    ),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)


In [21]:
ew4_imerg_norm_segment = petpipe.Pipeline(
    pyearthtools.pipeline.operations.xarray.normalisation.MagicNorm(cache_dir=imerg_norm_dir),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)


The last part is to convert to numpy arrays and cache those, so we can load the data as quickly as possible for training with pytorch.

In [22]:
ew4_imerg_ml = petpipe.Pipeline(
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    pyearthtools.pipeline.modifications.Cache( ew4_imerg_ml_prepped_dir,
                                               pattern_kwargs={'extension': 'npy'},
                                             ),
)

We now iterate through the data to calculate norms and populate the cache.

In [23]:
ew4_imerg_pipe = ew4_imerg_prep | ew4_imerg_norm_segment | ew4_imerg_ml

In [24]:
train_range = petpipe.Pipeline(
    petpipe.modifications.TemporalWindow(prior_indexes=[0,], posterior_indexes=[0,], timedelta=TimeDelta('30 minutes')),
    iterator=petpipe.iterators.DateRange('20250501T00', '20250701T00', interval='1 hour').randomise(), 
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)
val_range = petpipe.Pipeline(
    petpipe.modifications.TemporalWindow(prior_indexes=[0,], posterior_indexes=[0,], timedelta=TimeDelta('30 minutes')),
    iterator=petpipe.iterators.DateRange('20250701T00', '20250801T00', interval='1 hour'), 
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)

Calculated indexes


In [28]:
ew4_imerg_train_pipe = ew4_imerg_pipe | train_range
ew4_imerg_val_pipe = ew4_imerg_pipe | val_range